# Qwen2.5 × BBQ bias sweep (GPU)

Runs the four-size **Qwen2.5** family (0.5B / 1.5B / 3B / 7B) on BBQ in two scoring
modes — `text` and `letter --permute` — on **CUDA**, writing per-item CSVs to
`/kaggle/working/results/` (saved as notebook output).

**Before running** (right-hand *Settings* panel):
- **Accelerator = GPU** (a single 16 GB T4 or P100 is enough).
- **Internet = On** (needed for pip + model/dataset downloads).
- Optional: add a **Kaggle Secret** named `HF_TOKEN` (*Add-ons → Secrets*) to
  avoid Hugging Face download rate limits. Qwen2.5 is ungated, so it is optional.

See the repo README section "Run the sweep on Kaggle (GPU)" for the full walkthrough.

In [ ]:
# 1. Clone the pipeline into an ephemeral dir (keeps /kaggle/working output = CSVs only).
!rm -rf /tmp/bias-scaling
!git clone --depth 1 https://github.com/manitawtani74/bias-scaling.git /tmp/bias-scaling

In [ ]:
# 2. Install deps. Kaggle already ships CUDA-enabled torch — do NOT reinstall it
# (pip can pull a CPU build). Only upgrade the libraries the pipeline needs.
!pip install -q -U transformers datasets accelerate

In [ ]:
# 3. Optional HF token from a Kaggle Secret named HF_TOKEN.
import os
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
    print("HF token loaded from Kaggle secret.")
except Exception as e:
    print("No HF token (fine — Qwen2.5 is ungated):", e)

In [ ]:
# 4. Confirm the GPU is visible before spending time on the sweep.
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 5. Run the sweep: 4 sizes × 2 modes = 8 evaluations, all on CUDA.
import sys, subprocess

REPO = "/tmp/bias-scaling"
OUT = "/kaggle/working/results"
os.makedirs(OUT, exist_ok=True)

# (size, dtype). float32 up to 3B to match the CPU-baseline methodology;
# bfloat16 for 7B so it fits in 16 GB. If 3B OOMs on a 16 GB GPU, change its
# dtype to "bfloat16" here as well.
SIZES = [("0.5B", "float32"), ("1.5B", "float32"), ("3B", "float32"), ("7B", "bfloat16")]
TAG = {"0.5B": "05b", "1.5B": "15b", "3B": "3b", "7B": "7b"}

# (mode label, extra CLI args). text = default scoring; letterperm = letter mode
# with 6-way answer-order permutation debiasing.
MODES = [("text", []), ("letterperm", ["--scoring", "letter", "--permute"])]

for size, dtype in SIZES:
    model = f"Qwen/Qwen2.5-{size}"
    for mode, extra in MODES:
        out = f"{OUT}/qwen{TAG[size]}_{mode}_cuda.csv"
        cmd = [sys.executable, "-m", "src.evaluate",
               "--model", model, "--device", "cuda", "--dtype", dtype,
               "--sample", "200", "--seed", "0", "--output", out] + extra
        print("\n>>>", " ".join(cmd), flush=True)
        try:
            subprocess.run(cmd, cwd=REPO, check=True)
        except subprocess.CalledProcessError as e:
            print(f"!! {model} [{mode}] FAILED (exit {e.returncode}); continuing.", flush=True)

In [ ]:
# 6. List the CSV outputs (everything under /kaggle/working is saved on commit).
import glob
print("CSV outputs in /kaggle/working/results:")
for f in sorted(glob.glob("/kaggle/working/results/*.csv")):
    print(f"  {os.path.getsize(f):>10,d}  {os.path.basename(f)}")